In [1]:
from pyspark.sql import SparkSession
import numpy as np

### Session

In [2]:
spark = SparkSession.builder.appName("fundamentals").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/21 10:34:22 WARN Utils: Your hostname, Sarveshs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.24 instead (on interface en0)
26/01/21 10:34:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/21 10:34:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Read CSV

In [3]:
df = spark.read.csv("./datasets/uber.csv", header=True, inferSchema=True)
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- key: timestamp (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- passenger_count: integer (nullable = true)



### Print

In [4]:
df.show(n=5)

+--------+-------------------+-----------+-------------------+------------------+-----------------+------------------+-----------------+---------------+
|     _c0|                key|fare_amount|    pickup_datetime|  pickup_longitude|  pickup_latitude| dropoff_longitude| dropoff_latitude|passenger_count|
+--------+-------------------+-----------+-------------------+------------------+-----------------+------------------+-----------------+---------------+
|24238194|2015-05-07 19:52:06|        7.5|2015-05-08 01:22:06|-73.99981689453125|40.73835372924805|   -73.99951171875|40.72321701049805|              1|
|27835199|2009-07-17 20:04:56|        7.7|2009-07-18 01:34:56|        -73.994355|        40.728225|         -73.99471|        40.750325|              1|
|44984355|2009-08-24 21:45:00|       12.9|2009-08-25 03:15:00|        -74.005043|         40.74077|        -73.962565|        40.772647|              1|
|25894730|2009-06-26 08:22:21|        5.3|2009-06-26 13:52:21|        -73.976124| 

In [5]:
df.head(5)

[Row(_c0=24238194, key=datetime.datetime(2015, 5, 7, 19, 52, 6), fare_amount=7.5, pickup_datetime=datetime.datetime(2015, 5, 8, 1, 22, 6), pickup_longitude=-73.99981689453125, pickup_latitude=40.73835372924805, dropoff_longitude=-73.99951171875, dropoff_latitude=40.72321701049805, passenger_count=1),
 Row(_c0=27835199, key=datetime.datetime(2009, 7, 17, 20, 4, 56), fare_amount=7.7, pickup_datetime=datetime.datetime(2009, 7, 18, 1, 34, 56), pickup_longitude=-73.994355, pickup_latitude=40.728225, dropoff_longitude=-73.99471, dropoff_latitude=40.750325, passenger_count=1),
 Row(_c0=44984355, key=datetime.datetime(2009, 8, 24, 21, 45), fare_amount=12.9, pickup_datetime=datetime.datetime(2009, 8, 25, 3, 15), pickup_longitude=-74.005043, pickup_latitude=40.74077, dropoff_longitude=-73.962565, dropoff_latitude=40.772647, passenger_count=1),
 Row(_c0=25894730, key=datetime.datetime(2009, 6, 26, 8, 22, 21), fare_amount=5.3, pickup_datetime=datetime.datetime(2009, 6, 26, 13, 52, 21), pickup_long

### Data Types

In [6]:
print(df.dtypes)

[('_c0', 'int'), ('key', 'timestamp'), ('fare_amount', 'double'), ('pickup_datetime', 'timestamp'), ('pickup_longitude', 'double'), ('pickup_latitude', 'double'), ('dropoff_longitude', 'double'), ('dropoff_latitude', 'double'), ('passenger_count', 'int')]


### Describe

In [7]:
df.describe().show()

+-------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|summary|                 _c0|       fare_amount|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|   passenger_count|
+-------+--------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|  count|              200000|            200000|            200000|            200000|            199999|            199999|            200000|
|   mean|    2.771250368235E7|11.359955249999953|-72.52763791623703| 39.93588537801224| -72.5252916274742|  39.9238904018327|          1.684535|
| stddev|1.6013822212829249E7|  9.90177622506988| 11.43778733893171| 7.720539407361833| 13.11740777853664| 6.794828840545128|1.3859965507558791|
|    min|                   1|             -52.0|       -1340.64841|-74.01551500000001|        -3356.6663|-881.9855130000001|     

### Add Column

In [8]:
df = df.withColumn("pickup", (df["pickup_longitude"] + df["pickup_latitude"]) / 2)
df = df.withColumn("dropoff", (df["dropoff_longitude"] + df["dropoff_latitude"]) / 2)
df.show(n=5)

+--------+-------------------+-----------+-------------------+------------------+-----------------+------------------+-----------------+---------------+-----------------+-------------------+
|     _c0|                key|fare_amount|    pickup_datetime|  pickup_longitude|  pickup_latitude| dropoff_longitude| dropoff_latitude|passenger_count|           pickup|            dropoff|
+--------+-------------------+-----------+-------------------+------------------+-----------------+------------------+-----------------+---------------+-----------------+-------------------+
|24238194|2015-05-07 19:52:06|        7.5|2015-05-08 01:22:06|-73.99981689453125|40.73835372924805|   -73.99951171875|40.72321701049805|              1|-16.6307315826416|-16.638147354125977|
|27835199|2009-07-17 20:04:56|        7.7|2009-07-18 01:34:56|        -73.994355|        40.728225|         -73.99471|        40.750325|              1|       -16.633065|        -16.6221925|
|44984355|2009-08-24 21:45:00|       12.9|200

### Drop Column

In [10]:
df = df.drop(
    "_c0",
    "key",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
)
df.show(n=5)

+-----------+---------------+-----------------+-------------------+
|fare_amount|passenger_count|           pickup|            dropoff|
+-----------+---------------+-----------------+-------------------+
|        7.5|              1|-16.6307315826416|-16.638147354125977|
|        7.7|              1|       -16.633065|        -16.6221925|
|       12.9|              1|      -16.6321365|         -16.594959|
|        5.3|              3|        -16.59264|-16.580983500000002|
|       16.0|              5|       -16.590469|-16.605917499999997|
+-----------+---------------+-----------------+-------------------+
only showing top 5 rows


### Rename Column

In [11]:
df = df.withColumnRenamed("passenger_count", "count")
df.show(5)

+-----------+-----+-----------------+-------------------+
|fare_amount|count|           pickup|            dropoff|
+-----------+-----+-----------------+-------------------+
|        7.5|    1|-16.6307315826416|-16.638147354125977|
|        7.7|    1|       -16.633065|        -16.6221925|
|       12.9|    1|      -16.6321365|         -16.594959|
|        5.3|    3|        -16.59264|-16.580983500000002|
|       16.0|    5|       -16.590469|-16.605917499999997|
+-----------+-----+-----------------+-------------------+
only showing top 5 rows
